In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.preprocessing import StandardScaler

In [ ]:
fonts_df = pd.read_csv("./csv/fonts_384_p5_raw.csv")
fonts_df = fonts_df.dropna(axis=1)

num_chars, _ = fonts_df.shape
char_list = np.sort(fonts_df["char"].unique()).tolist()

# fonts_df.round(6).to_csv("./csv/fonts_384_raw.csv", index=False)
fonts_df

In [ ]:
# Center each letter on 0,0
x_points = fonts_df.iloc[:, 2::2].values
y_points = -fonts_df.iloc[:, 3::2].values

x_avg = (x_points.max(axis=1) + x_points.min(axis=1)) / 2
y_avg = (y_points.max(axis=1) + y_points.min(axis=1)) / 2

x_points_centered = x_points - x_avg.reshape(-1,1)
y_points_centered = y_points - y_avg.reshape(-1,1)

font_points_xy = np.stack((x_points_centered, y_points_centered), axis=2)

font_points_df = pd.DataFrame(font_points_xy.reshape(num_chars, -1), columns=fonts_df.columns[2:])
font_points_df

In [ ]:
font_points_centered_df = fonts_df[["font", "char"]].join(font_points_df)
# font_points_centered_df.round(6).to_csv("./csv/fonts_384_centered.csv", index=False)
font_points_centered_df

In [ ]:
# get polar heuristic per x,y point
def polar_order(xy):
  x,y=xy
  r = np.sqrt(x**2 + y**2)
  a = np.arctan2(y, x)
  return 100*a + r

font_polar = np.apply_along_axis(polar_order, axis=2, arr=font_points_xy)

In [ ]:
# re-order (x,y)s based on polar heuristic along axis=1
row_idxs = np.indices(font_polar.shape)[0].reshape(-1)
col_idxs = font_polar.argsort(axis=1).reshape(-1)
font_points_xy_sorted = font_points_xy[row_idxs, col_idxs].reshape(font_points_xy.shape)

# back to x,y,x,y,x,y,x,y,...
font_points_sorted_df = pd.DataFrame(font_points_xy_sorted.reshape(num_chars, -1), columns=font_points_df.columns)

In [ ]:
font_points_sorted_out_df = fonts_df[["font", "char"]].join(font_points_sorted_df)
# font_points_sorted_out_df.round(6).to_csv("./csv/fonts_384_sorted.csv", index=False)
font_points_sorted_out_df

## Read Data and explore PCA

In [ ]:
fonts_centered_in_df = pd.read_csv("./csv/fonts_384_centered.csv")
fonts_sorted_in_df = pd.read_csv("./csv/fonts_384_sorted.csv")

char_list = np.sort(fonts_sorted_in_df["char"].unique()).tolist()

font_points_centered_df = fonts_centered_in_df.drop(columns=["font", "char"])
font_points_sorted_df = fonts_sorted_in_df.drop(columns=["font", "char"])
font_points_sorted_df

In [ ]:
font_idx = 0
letter_idx = char_list.index('A')
df_idx = letter_idx + font_idx*(len(char_list))

xs = font_points_centered_df.iloc[df_idx, 0::2]
ys = font_points_centered_df.iloc[df_idx, 1::2]

xs_sorted = font_points_sorted_df.iloc[df_idx, 0::2]
ys_sorted = font_points_sorted_df.iloc[df_idx, 1::2]

plt.plot(xs, ys, marker="o", linestyle="", markersize=4)
plt.xlim([-20,20])
plt.ylim([-20,20])
plt.show()

plt.plot(xs_sorted, ys_sorted, marker="o", linestyle="", markersize=4)
plt.xlim([-20,20])
plt.ylim([-20,20])
plt.show()

In [ ]:
mpca = PCA(n_components=len(char_list))

# font_pca = mpca.fit_transform(font_points_centered_df)
font_pca = mpca.fit_transform(font_points_sorted_df)
recon_font = mpca.inverse_transform(font_pca)

print(sum(mpca.explained_variance_ratio_), mpca.n_components)

pca_dists = euclidean_distances(font_pca, font_pca)
pca_dists_sorted = pca_dists.argsort(axis=1)

In [ ]:
font_idx = 0
letter_idx = char_list.index('A')
df_idx = letter_idx + font_idx*(len(char_list))

xs = recon_font[df_idx, 0::2]
ys = recon_font[df_idx, 1::2]

plt.scatter(xs, ys)
plt.xlim([-20,20])
plt.ylim([-20,20])
plt.show()

In [ ]:
font_idx = 0
letter_idx = char_list.index('A')
df_idx = letter_idx + font_idx * (len(char_list))

for idx in pca_dists_sorted[df_idx, :8]:
  xs = font_points_sorted_df.iloc[idx, 0::2]
  ys = font_points_sorted_df.iloc[idx, 1::2]

  plt.scatter(xs, ys)
  plt.xlim([-20,20])
  plt.ylim([-20,20])
  plt.show()